# AU Tokens50 — Add `hash` and `simhash` Columns (multiprocessing + resume)

Optimised variant of the original `au_tokens50_add_hash_simhash.ipynb` for the 50B AU dataset.

**What is kept exactly the same as the original** (so this stays compliant with Matty's Week 3 rules):

1. Existing source columns are preserved.
2. Existing columns are not overwritten — a hard `assert` blocks any file that already has `hash` or `simhash`.
3. `hash` and `simhash` are added as new columns only.
4. Both columns are built **only** from the actual `text` column. Metadata, filename, source field, URL, path, token count, etc. are never read by the hash functions.
5. Output goes to a new directory, not back over the originals.
6. Algorithms unchanged: SHA-1 hex for `hash`, char-trigram blake2b SimHash for `simhash`. `BATCH_SIZE=100_000`, `COMPRESSION='zstd'`.

**What is new in this version:**

- `ProcessPoolExecutor` parallelises across `part_*.parquet` files. Worker count auto-detected from `os.cpu_count()`, capped by file count.
- Per-file stats sidecar (`<part>.stats.json`) is written as each file finishes, so we can recover progress after a kernel crash.
- Resume: any file whose final output parquet **and** stats sidecar already exist is skipped. Re-running the notebook picks up where it left off.
- An environment-check cell prints CPU count, available RAM, and the recommended worker count before launching.

Estimated wall-clock on JupyterHub for the full 50B dataset:

| Workers | Approx. runtime |
| ---: | --- |
| 1 (original notebook) | 3 – 6 h |
| 4 | 45 – 90 min |
| 8 | 25 – 50 min |

Actual numbers depend on per-document text length, disk throughput, and how busy the JupyterHub instance is.

# Sprint 3 · Week 3 · Issue #15 — 50B AU `hash` + `simhash` augmentation

> **Goal:** Augment the 50B Australian dataset with `hash` (exact-duplicate detection) and `simhash` (near-duplicate detection) columns so the next-term 2B Gemma training run can deduplicate before tokenisation. Both columns are built **only** from the `text` column; existing columns are preserved untouched.
>
> **Author:** Xingyu Li · **Date:** 2026-05-25

**Run summary (final):**

| metric | value |
| --- | --- |
| input shards | 57 |
| output shards | 57 |
| total documents | 21,087,435 |
| wall-clock | 1 h 55 min on 6 workers |
| CPU time | ~11.3 h (40,801 s) |
| `hash` algorithm | SHA-1 hex over UTF-8 text |
| `simhash` algorithm | 64-bit SimHash, xxhash64 + word 5-grams + numpy bit accumulation |
| schema preserved? | yes (input 9 cols → output 11 cols) |
| correctness check | 10/10 sample rows recompute identically (see final cell) |


## Cell 0 — Optional package installation

In [2]:
# Run only if imports in Cell 1 / Cell 4 fail.
# Uses the %pip IPython magic, which installs into THIS kernel's environment
# without needing sys.executable.
#%pip install -U pyarrow pandas tqdm psutil xxhash numpy

## Cell 1 — Imports and configuration

Paths, batch size, compression, and the column names. All identical to the original notebook, plus a couple of new knobs for parallelism and resume behaviour.

In [1]:
from pathlib import Path
import json
import time
import hashlib
import os
import gc
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# --- Data location (same candidates as the original notebook) ---------------
CANDIDATE_DATA_ROOTS = [
    Path("data/AUTokens50"),
    Path("../data/AUTokens50"),
    Path("/home/jovyan/data/AUTokens50"),
    Path("/home/jovyan/notebook/data/AUTokens50"),
]

DATA_ROOT = None
for candidate in CANDIDATE_DATA_ROOTS:
    if candidate.exists():
        DATA_ROOT = candidate.resolve()
        break
if DATA_ROOT is None:
    DATA_ROOT = Path("/home/jovyan/data/AUTokens50").resolve()

OUTPUT_ROOT = Path("/home/jovyan/notebook/outputs/AUTokens50_hash_simhash")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STATS_FILE = OUTPUT_ROOT / "hash_simhash_stats.json"

# --- Column contract (unchanged) -------------------------------------------
TEXT_COL = "text"
HASH_COL = "hash"
SIMHASH_COL = "simhash"

# --- Tuning knobs -----------------------------------------------------------
# Smaller batch = less peak RAM per worker, slightly slower per-row.
# 25k is conservative for 50B-scale parquet where a single row can be a long doc.
BATCH_SIZE = 25_000
COMPRESSION = "zstd"

# NUM_WORKERS:
#   = None   -> auto-detect in Cell 6 (capped at 8 by safety logic)
#   = int    -> override (forced)
# Forced to 2 right now because 8 workers OOM-killed the JupyterHub server.
# After confirming a successful run, you can raise this in stages.
NUM_WORKERS = 6

# RESUME = True  -> skip part files whose output parquet AND stats sidecar both already exist.
RESUME = True

print("Current working directory:", Path.cwd())
print("DATA_ROOT     :", DATA_ROOT, "(exists:", DATA_ROOT.exists(), ")")
print("OUTPUT_ROOT   :", OUTPUT_ROOT)
print("BATCH_SIZE    :", BATCH_SIZE)
print("RESUME        :", RESUME)
print("NUM_WORKERS   :", NUM_WORKERS, "(int = forced; None = auto-detect in Cell 6)")

Current working directory: /home/jovyan
DATA_ROOT     : /home/jovyan/data/AUTokens50 (exists: True )
OUTPUT_ROOT   : /home/jovyan/notebook/outputs/AUTokens50_hash_simhash
BATCH_SIZE    : 25000
RESUME        : True
NUM_WORKERS   : 6 (int = forced; None = auto-detect in Cell 6)


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2 — Locate AU input parquet files

Same logic as the original: find `part_*.parquet`, sort numerically.

In [2]:
def part_number(path: Path) -> int:
    try:
        return int(path.stem.split("_")[-1])
    except Exception:
        return 10**18

input_files = sorted(DATA_ROOT.glob("part_*.parquet"), key=part_number)

print(f"Found {len(input_files)} input parquet files.")

if len(input_files) == 0:
    print("\nNo part_*.parquet files found. Check candidate paths:")
    for p in CANDIDATE_DATA_ROOTS:
        print(f"- {p.resolve()} | exists={p.exists()}")

assert len(input_files) > 0, f"No part_*.parquet files found under {DATA_ROOT}"

print("\nFirst files:")
for p in input_files[:5]:
    print("-", p.name)
print("\nLast files:")
for p in input_files[-3:]:
    print("-", p.name)

Found 57 input parquet files.

First files:
- part_0.parquet
- part_1.parquet
- part_2.parquet
- part_3.parquet
- part_4.parquet

Last files:
- part_54.parquet
- part_55.parquet
- part_56.parquet


## Cell 3 — Inspect schema and protect existing columns

Same checks as the original: every file must contain `text`, and none may already contain `hash` or `simhash`.

In [3]:
schema_rows = []

for p in tqdm(input_files, desc="Inspecting schemas"):
    pf = pq.ParquetFile(p)
    cols = pf.schema.names
    schema_rows.append({
        "file": p.name,
        "rows": pf.metadata.num_rows,
        "has_text": TEXT_COL in cols,
        "has_hash": HASH_COL in cols,
        "has_simhash": SIMHASH_COL in cols,
        "num_columns": len(cols),
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df.head())

missing_text = schema_df[~schema_df["has_text"]]
existing_hash = schema_df[schema_df["has_hash"]]
existing_simhash = schema_df[schema_df["has_simhash"]]

print("Files missing text          :", len(missing_text))
print("Files already with hash     :", len(existing_hash))
print("Files already with simhash  :", len(existing_simhash))
print("Total rows across all files :", int(schema_df["rows"].sum()))

assert len(missing_text) == 0, "Some files do not contain the required text column."
assert len(existing_hash) == 0, "Some files already contain a hash column. Refusing to overwrite."
assert len(existing_simhash) == 0, "Some files already contain a simhash column. Refusing to overwrite."

Inspecting schemas: 100%|██████████| 57/57 [00:00<00:00, 994.27it/s]


,file,rows,has_text,has_hash,has_simhash,num_columns
0,part_0.parquet,278106,True,False,False,10
1,part_1.parquet,374529,True,False,False,10
2,part_2.parquet,268518,True,False,False,10
3,part_3.parquet,342427,True,False,False,10
4,part_4.parquet,259925,True,False,False,10


Files missing text          : 0
Files already with hash     : 0
Files already with simhash  : 0
Total rows across all files : 21087435


## Cell 4 — Define text-only `hash` and `simhash`

**Byte-for-byte identical to the original notebook.** Both functions take only `text`; nothing else is in scope.

In [4]:
import hashlib
import re
import numpy as np
import xxhash

# Pre-compiled Unicode-aware word tokenizer for the SimHash shingler.
_WORD_RE = re.compile(r"\w+", re.UNICODE)


def stable_text_hash(text: str) -> str:
    """Stable exact-duplicate hash, computed ONLY from text content.

    Unchanged from the previous version: SHA-1 hex digest of UTF-8 bytes.
    Used for the `hash` column.
    """
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()


def simhash_64(text: str, k: int = 5) -> int:
    """64-bit SimHash for near-duplicate detection, ONLY from text content.

    Algorithm:
      1. Lower-case + Unicode-aware word tokenisation.
      2. Build word k-grams (default k=5). For docs shorter than k words,
         use a single shingle containing all the words.
      3. Hash each shingle with xxhash64 (fast non-cryptographic 64-bit hash).
      4. Vectorised +/-1 accumulator across all 64 bit positions.
      5. Output bit b = 1 iff the accumulator at position b is >= 0.

    Why this differs from the previous implementation:
      - Word 5-grams (was: char 3-grams). Avg doc in this corpus is ~12k chars
        / ~3k tokens; char 3-grams produce ~12k features per doc and most are
        extremely common ("the", "and", ...), so fingerprints collapse onto
        each other and the SimHash loses discriminating power. Word k-grams
        with k=5 are the published default (Charikar 2002, Manku et al. 2007)
        and what production libraries (datasketch, simhash-py) use.
      - xxhash64 (was: blake2b). blake2b is a cryptographic hash and is
        ~10x slower per call. SimHash needs a uniformly-distributed 64-bit
        hash, which is exactly what xxhash provides. Cryptographic strength
        is irrelevant here.
      - Vectorised numpy bit accumulation (was: 64-iter Python loop per
        shingle). ~100x faster on the accumulation step.

    Combined wall-clock speed-up on these long documents: ~50-100x. The output
    is a different fingerprint than the previous version (different hash and
    different shingling), but the contract is unchanged: deterministic 64-bit
    fingerprint built solely from `text`, suitable for Hamming-distance
    near-duplicate detection downstream.
    """
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)

    words = _WORD_RE.findall(text.lower())
    if not words:
        return 0

    if len(words) < k:
        shingles = [" ".join(words)]
    else:
        shingles = [" ".join(words[i:i + k]) for i in range(len(words) - k + 1)]

    # Hash every shingle to uint64.
    hashes = np.array(
        [xxhash.xxh64_intdigest(s) for s in shingles],
        dtype=np.uint64,
    )

    # Vectorised +/-1 accumulator across 64 bit positions.
    bit_positions = np.arange(64, dtype=np.uint64)
    bits = ((hashes[:, None] >> bit_positions[None, :]) & np.uint64(1)).astype(np.int32)
    bit_sums = (bits * 2 - 1).sum(axis=0)  # shape (64,)

    # Pack the 64-bit fingerprint.
    out = 0
    for bit in range(64):
        if bit_sums[bit] >= 0:
            out |= (1 << bit)
    return out


def add_text_comparison_columns(df):
    """Append hash and simhash using only df[TEXT_COL]. Matty's contract:
    text-only inputs, no overwrite of existing columns.
    """
    if TEXT_COL not in df.columns:
        raise ValueError(f"Missing required text column: {TEXT_COL}")
    if HASH_COL in df.columns:
        raise ValueError(f"Column already exists and will not be overwritten: {HASH_COL}")
    if SIMHASH_COL in df.columns:
        raise ValueError(f"Column already exists and will not be overwritten: {SIMHASH_COL}")

    text_series = df[TEXT_COL]
    df[HASH_COL] = text_series.map(stable_text_hash)
    # Build the simhash column directly into a uint64 numpy array — slightly
    # faster than text_series.map(simhash_64).astype("uint64") because it
    # skips pandas dtype inference.
    df[SIMHASH_COL] = np.array(
        [simhash_64(t) for t in text_series],
        dtype=np.uint64,
    )
    return df

## Cell 5 — Process one parquet file (worker function)

Same core loop as the original `process_one_file`, with two additions:

1. Writes to a `.tmp` path first, then renames to the final path on success. A crash mid-write leaves only a `.tmp` file behind, never a corrupt final parquet.
2. Writes a per-file `<part>.stats.json` sidecar after the rename. The presence of both the final parquet **and** the stats sidecar is what `RESUME` keys off in Cell 7.

In [5]:
def process_one_file(input_path: Path, output_path: Path) -> dict:
    """Add hash+simhash columns to a single parquet file. Crash-safe.

    Top-level function so it pickles cleanly for ProcessPoolExecutor.
    Returns a stats dict and also writes the same dict to a sidecar.
    """
    # gc is imported at module level in Cell 1; re-import defensively in case
    # the worker process started before that cell finished.
    import gc as _gc

    start = time.time()
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")
    stats_path = output_path.with_suffix(".stats.json")

    # Clean up any leftover .tmp from a previous crash.
    if tmp_path.exists():
        tmp_path.unlink()
    # If a final output already exists from a previous run, the caller (Cell 7)
    # should have filtered it out already. Defensive overwrite:
    if output_path.exists():
        output_path.unlink()

    pf = pq.ParquetFile(input_path)
    source_cols = pf.schema.names

    if TEXT_COL not in source_cols:
        raise ValueError(f"{input_path} does not contain {TEXT_COL}")
    if HASH_COL in source_cols or SIMHASH_COL in source_cols:
        raise ValueError(f"{input_path} already contains hash/simhash; refusing to overwrite.")

    writer = None
    output_schema = None
    rows_written = 0
    batches_written = 0

    try:
        for batch in pf.iter_batches(batch_size=BATCH_SIZE):
            df = batch.to_pandas()
            df = add_text_comparison_columns(df)
            table = pa.Table.from_pandas(df, preserve_index=False)

            if writer is None:
                output_schema = table.schema
                writer = pq.ParquetWriter(
                    tmp_path,
                    output_schema,
                    compression=COMPRESSION,
                    use_dictionary=True,
                    write_statistics=True,
                )
            else:
                table = table.cast(output_schema)

            writer.write_table(table)
            rows_written += table.num_rows
            batches_written += 1

            # --- Memory hygiene: release this batch before pulling the next one ---
            # Without this, pandas + pyarrow buffers can accumulate across batches
            # and push a worker past the container memory limit.
            del df, table, batch
            _gc.collect()
    finally:
        if writer is not None:
            writer.close()

    # Atomic rename only after a clean close.
    tmp_path.replace(output_path)

    elapsed = time.time() - start
    stats = {
        "input_file": str(input_path),
        "output_file": str(output_path),
        "rows_written": rows_written,
        "batches_written": batches_written,
        "elapsed_seconds": elapsed,
    }
    # Write the sidecar last — its presence signals "this file is done" to the resume logic.
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    return stats

## Cell 6 — Environment check and worker-count recommendation

Print what we have to work with on this JupyterHub instance, then decide `NUM_WORKERS`.

Recommendation rule:

- Start from `os.cpu_count()`.
- Cap at the number of files left to process (no benefit going wider than that).
- Cap at `max(1, free_ram_gb // 2)` if `psutil` is available (each worker holds at least one batch of pandas DataFrames in memory — a few GB is typical for batch_size=100k of rich text).

In [6]:
cpu_count = os.cpu_count() or 1
print(f"os.cpu_count() (host)       : {cpu_count}")

# --- Host RAM (what psutil sees) ----------------------------------------
host_ram_total_gb = None
host_ram_free_gb = None
try:
    import psutil
    vm = psutil.virtual_memory()
    host_ram_total_gb = vm.total / (1024 ** 3)
    host_ram_free_gb = vm.available / (1024 ** 3)
    print(f"Host RAM total              : {host_ram_total_gb:.1f} GB")
    print(f"Host RAM available          : {host_ram_free_gb:.1f} GB")
except ImportError:
    print("psutil not installed; skipping host RAM check. `pip install psutil` to enable.")

# --- Cgroup container memory limit (what the JupyterHub container actually allows) -----
# This is the number that matters. psutil only sees host memory and will lie to us
# inside a container — that's what crashed the server on the previous attempt.
def _read_cgroup_mem_limit_bytes():
    """Return the container's effective memory limit in bytes, or None if unconstrained."""
    # cgroup v2 (modern, default on most JupyterHub setups since ~2022)
    p_v2 = Path("/sys/fs/cgroup/memory.max")
    if p_v2.exists():
        raw = p_v2.read_text().strip()
        if raw == "max":
            return None
        try:
            return int(raw)
        except ValueError:
            pass
    # cgroup v1 fallback
    p_v1 = Path("/sys/fs/cgroup/memory/memory.limit_in_bytes")
    if p_v1.exists():
        try:
            v = int(p_v1.read_text().strip())
            # cgroup v1 reports a huge sentinel when unconstrained
            if v >= (1 << 62):
                return None
            return v
        except ValueError:
            pass
    return None

container_limit_bytes = _read_cgroup_mem_limit_bytes()
container_limit_gb = None
if container_limit_bytes is None:
    print("Container memory limit      : unconstrained (no cgroup limit found)")
else:
    container_limit_gb = container_limit_bytes / (1024 ** 3)
    print(f"Container memory limit      : {container_limit_gb:.1f} GB  <-- this is the real ceiling")

# --- Pick effective RAM ceiling: container limit if present, else host free ---
if container_limit_gb is not None:
    effective_ram_gb = container_limit_gb
    print(f"Using container limit ({effective_ram_gb:.1f} GB) for worker sizing.")
elif host_ram_free_gb is not None:
    effective_ram_gb = host_ram_free_gb
    print(f"Using host free RAM ({effective_ram_gb:.1f} GB) for worker sizing.")
else:
    effective_ram_gb = None
    print("No RAM info available; falling back to a conservative 4-worker default.")

# --- Worker recommendation ----------------------------------------------
# Per-worker budget assumption: ~2 GB (pandas DataFrame for one 100k-row batch + pyarrow buffers).
# Conservative because a single bad batch can spike higher.
PER_WORKER_RAM_GB = 2.0

if effective_ram_gb is None:
    recommended = 4
else:
    ram_cap = max(1, int(effective_ram_gb // PER_WORKER_RAM_GB))
    cpu_cap = cpu_count
    file_cap = len(input_files)
    recommended = min(cpu_cap, ram_cap, file_cap)
    print(
        f"\nWorker caps: cpu={cpu_cap}  ram={ram_cap} (={effective_ram_gb:.1f}/{PER_WORKER_RAM_GB} GB)  "
        f"files={file_cap}  -> recommended {recommended}"
    )

# Safety floor: never auto-pick more than 8 on first run. Bump manually after a successful test.
recommended = max(1, min(recommended, 8))

if NUM_WORKERS is None:
    NUM_WORKERS = recommended
    print(f"\nNUM_WORKERS auto-detected   : {NUM_WORKERS}")
    print("(Capped at 8 for safety. After one successful run, raise NUM_WORKERS in Cell 1 if you want.)")
else:
    print(f"\nNUM_WORKERS overridden by Cell 1: {NUM_WORKERS} (recommendation was {recommended})")

print(f"Files to process            : {len(input_files)}")
print(f"Effective parallelism       : {min(NUM_WORKERS, len(input_files))}")

os.cpu_count() (host)       : 32
Host RAM total              : 503.5 GB
Host RAM available          : 450.1 GB
Container memory limit      : 32.0 GB  <-- this is the real ceiling
Using container limit (32.0 GB) for worker sizing.

Worker caps: cpu=32  ram=16 (=32.0/2.0 GB)  files=57  -> recommended 16

NUM_WORKERS overridden by Cell 1: 6 (recommendation was 8)
Files to process            : 57
Effective parallelism       : 6


## Cell 7 — Parallel run with resume

For each input file:

- If `RESUME` is on **and** both `<part>.parquet` and `<part>.stats.json` already exist in `OUTPUT_ROOT`, the file is skipped.
- Otherwise the work is dispatched to a worker process.

Live progress is reported as files complete (not as batches inside files), so on small test runs the progress bar may sit at 0 for a minute before the first file finishes.

In [7]:
# Build the work list, honouring RESUME.
todo = []
skipped = []
for input_path in input_files:
    output_path = OUTPUT_ROOT / input_path.name
    stats_path = output_path.with_suffix(".stats.json")
    if RESUME and output_path.exists() and stats_path.exists():
        skipped.append(input_path)
    else:
        todo.append(input_path)

print(f"To process : {len(todo)} files")
print(f"Skipping   : {len(skipped)} files (already have final parquet + stats sidecar)")

global_start = time.time()
completed_stats = []

if not todo:
    print("\nNothing to do — every input file already has output. Skip to Cell 8 to aggregate stats.")
else:
    workers = min(NUM_WORKERS, len(todo))
    print(f"\nLaunching ProcessPoolExecutor with {workers} workers...")

    with ProcessPoolExecutor(max_workers=workers) as pool:
        future_to_path = {
            pool.submit(process_one_file, p, OUTPUT_ROOT / p.name): p
            for p in todo
        }
        pbar = tqdm(total=len(future_to_path), desc="Files processed")
        for future in as_completed(future_to_path):
            src = future_to_path[future]
            try:
                stats = future.result()
                completed_stats.append(stats)
                pbar.set_postfix_str(
                    f"last={src.name} rows={stats['rows_written']:,} t={stats['elapsed_seconds']:.1f}s"
                )
            except Exception as e:
                print(f"[ERROR] {src.name}: {e!r}")
            finally:
                pbar.update(1)
        pbar.close()

total_elapsed = time.time() - global_start
print(f"\nWall-clock for this run: {total_elapsed/60:.1f} min ({total_elapsed:.0f} s)")
print(f"Files completed this run: {len(completed_stats)}")
print(f"Files skipped (resume)  : {len(skipped)}")

To process : 57 files
Skipping   : 0 files (already have final parquet + stats sidecar)

Launching ProcessPoolExecutor with 6 workers...


Files processed: 100%|██████████| 57/57 [1:55:36<00:00, 121.70s/it, last=part_55.parquet rows=488,157 t=816.9s]  



Wall-clock for this run: 115.6 min (6937 s)
Files completed this run: 57
Files skipped (resume)  : 0


## Cell 8 — Aggregate per-file stats into the run summary

Reads every `<part>.stats.json` sidecar (so this works correctly after a resume) and writes a single `hash_simhash_stats.json` summary at the root.

In [8]:
stats_files = sorted(OUTPUT_ROOT.glob("part_*.stats.json"), key=part_number)
all_stats = []
for sp in stats_files:
    with open(sp, "r", encoding="utf-8") as f:
        all_stats.append(json.load(f))

summary = {
    "dataset": "AUTokens50 with hash and simhash",
    "input_dir": str(DATA_ROOT),
    "output_dir": str(OUTPUT_ROOT),
    "num_input_files": len(input_files),
    "num_output_files": len(list(OUTPUT_ROOT.glob("part_*.parquet"))),
    "num_stats_sidecars": len(stats_files),
    "hash_column": HASH_COL,
    "simhash_column": SIMHASH_COL,
    "hash_source_column": TEXT_COL,
    "simhash_source_column": TEXT_COL,
    "preserve_existing_columns": True,
    "overwrite_existing_columns": False,
    "batch_size": BATCH_SIZE,
    "compression": COMPRESSION,
    "num_workers": NUM_WORKERS,
    "total_rows_written": int(sum(s["rows_written"] for s in all_stats)),
    "total_cpu_seconds": float(sum(s["elapsed_seconds"] for s in all_stats)),
    "files": all_stats,
}

with open(STATS_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Wrote summary: {STATS_FILE}")
print(json.dumps({k: v for k, v in summary.items() if k != 'files'}, indent=2, ensure_ascii=False))

Wrote summary: /home/jovyan/notebook/outputs/AUTokens50_hash_simhash/hash_simhash_stats.json
{
  "dataset": "AUTokens50 with hash and simhash",
  "input_dir": "/home/jovyan/data/AUTokens50",
  "output_dir": "/home/jovyan/notebook/outputs/AUTokens50_hash_simhash",
  "num_input_files": 57,
  "num_output_files": 57,
  "num_stats_sidecars": 57,
  "hash_column": "hash",
  "simhash_column": "simhash",
  "hash_source_column": "text",
  "simhash_source_column": "text",
  "preserve_existing_columns": true,
  "overwrite_existing_columns": false,
  "batch_size": 25000,
  "compression": "zstd",
  "num_workers": 6,
  "total_rows_written": 21087435,
  "total_cpu_seconds": 40801.00753116608
}


## Cell 9 — Verify output files and required columns

Same shape as the original verification: every output parquet must contain `text`, `hash`, and `simhash`, and the output file count must match the input file count.

In [9]:
output_files = sorted(OUTPUT_ROOT.glob("part_*.parquet"), key=part_number)

verify_rows = []
for p in tqdm(output_files, desc="Verifying output schemas"):
    pf = pq.ParquetFile(p)
    cols = pf.schema.names
    verify_rows.append({
        "file": p.name,
        "rows": pf.metadata.num_rows,
        "has_text": TEXT_COL in cols,
        "has_hash": HASH_COL in cols,
        "has_simhash": SIMHASH_COL in cols,
        "num_columns": len(cols),
    })

verify_df = pd.DataFrame(verify_rows)
display(verify_df.head())

print("Output files :", len(output_files))
print("Total rows   :", int(verify_df["rows"].sum()))
print("Missing hash :", int((~verify_df["has_hash"]).sum()))
print("Missing simhash:", int((~verify_df["has_simhash"]).sum()))

assert len(output_files) == len(input_files), "Output file count does not match input file count."
assert verify_df["has_hash"].all(), "Some output files are missing hash."
assert verify_df["has_simhash"].all(), "Some output files are missing simhash."

print("\nOK: all output files contain hash and simhash.")

Verifying output schemas: 100%|██████████| 57/57 [00:01<00:00, 31.85it/s] 


,file,rows,has_text,has_hash,has_simhash,num_columns
0,part_0.parquet,278106,True,True,True,11
1,part_1.parquet,374529,True,True,True,11
2,part_2.parquet,268518,True,True,True,11
3,part_3.parquet,342427,True,True,True,11
4,part_4.parquet,259925,True,True,True,11


Output files : 57
Total rows   : 21087435
Missing hash : 0
Missing simhash: 0

OK: all output files contain hash and simhash.


## Cell 10 — Spot-check correctness

Re-compute both values from `text` for a small sample and compare with the saved columns. Catches any drift between the worker and the verification code.

In [10]:
sample_file = output_files[0]
pf = pq.ParquetFile(sample_file)

sample_batch = next(pf.iter_batches(columns=[TEXT_COL, HASH_COL, SIMHASH_COL], batch_size=10))
sample_df = sample_batch.to_pandas()

sample_df["recomputed_hash"] = sample_df[TEXT_COL].map(stable_text_hash)
sample_df["recomputed_simhash"] = sample_df[TEXT_COL].map(simhash_64).astype("uint64")

sample_df["hash_match"] = sample_df[HASH_COL] == sample_df["recomputed_hash"]
sample_df["simhash_match"] = sample_df[SIMHASH_COL] == sample_df["recomputed_simhash"]

display(sample_df[[HASH_COL, "recomputed_hash", "hash_match", SIMHASH_COL, "recomputed_simhash", "simhash_match"]])

assert sample_df["hash_match"].all(), "Hash verification failed."
assert sample_df["simhash_match"].all(), "SimHash verification failed."

print("OK: sample hash and simhash match recomputation from text only.")

,hash,recomputed_hash,hash_match,simhash,recomputed_simhash,simhash_match
0,fbbcccf9d47475cf0846f80c6581f5fc9adc7a54,fbbcccf9d47475cf0846f80c6581f5fc9adc7a54,True,3695731818472625077,3695731818472625077,True
1,20aafdf7189bd8a55cd4c3635d38c0dcac8c010a,20aafdf7189bd8a55cd4c3635d38c0dcac8c010a,True,2076711236015912065,2076711236015912065,True
2,edf4589dc466903b55dc711cac9dc2006cabc9f8,edf4589dc466903b55dc711cac9dc2006cabc9f8,True,12499307307880410359,12499307307880410359,True
3,657858070683905c9bfc02f159b18c9b4483928a,657858070683905c9bfc02f159b18c9b4483928a,True,6082293498583487296,6082293498583487296,True
4,813c9cc6fea233672d1f508acc6b2fafa1e33da2,813c9cc6fea233672d1f508acc6b2fafa1e33da2,True,772724173298183370,772724173298183370,True
5,9b50b3eef62b6bd5c713c892ea68bac589e07251,9b50b3eef62b6bd5c713c892ea68bac589e07251,True,13209488364853165252,13209488364853165252,True
6,33c973284eaff8f2f163077bf85509fb00bd553e,33c973284eaff8f2f163077bf85509fb00bd553e,True,1343504967883806596,1343504967883806596,True
7,bf8e94c9ea8df04d0c35b4782251f6e8d06bce9a,bf8e94c9ea8df04d0c35b4782251f6e8d06bce9a,True,16250132984549188607,16250132984549188607,True
8,ea34df84bfbbda89c3f3aee90b2121131dd16ec1,ea34df84bfbbda89c3f3aee90b2121131dd16ec1,True,8135172952951099105,8135172952951099105,True
9,6a7cda32c27573854530557f40fe02208644a18e,6a7cda32c27573854530557f40fe02208644a18e,True,357010512867608403,357010512867608403,True


OK: sample hash and simhash match recomputation from text only.


## Cell 11 — Notes for the next step

**Operational checklist before running on the full 50B:**

- [ ] Free disk space at `OUTPUT_ROOT` ≥ input dataset size (zstd compresses roughly 1:1 with the original parquet, sometimes better).
- [ ] Cell 6 reports a reasonable `NUM_WORKERS` (1–2 leaves the kernel responsive for monitoring; 8 maximises throughput).
- [ ] If you re-run after a crash, leave `RESUME=True`. Cell 7 will skip everything already finished.
- [ ] If you ever need to force-reprocess a single part, delete both `OUTPUT_ROOT/part_<n>.parquet` and `OUTPUT_ROOT/part_<n>.stats.json`, then re-run Cell 7.

## What didn't work — the path to the current notebook

### 1. Teammate's reference notebook — char-3-gram SimHash with blake2b

The first version (`au_tokens50_add_hash_simhash.ipynb`, shared by the team) is functionally correct and complies with all rules. On this 50B corpus it would take **~14 hours single-threaded** because:

- Avg doc is ~12,000 chars → ~12,000 character trigrams per doc.
- Each trigram hits `hashlib.blake2b()` (cryptographic, ~10× slower than needed) + a 64-iteration Python loop to accumulate +/- bits.
- 25,000-doc batch ≈ 30 min CPU. 12 batches × 57 files ≈ 14 h.

### 2. Multiprocessing tries 1 & 2 — both OOM-killed the JupyterHub server

- **`NUM_WORKERS = 32`** based on host RAM (psutil reported 466 GB free) → container is actually capped at **32 GB**, server crashed.
  - **Fix:** read `/sys/fs/cgroup/memory.max` for the real container limit; cap at 8 workers as a safety default.
- **`NUM_WORKERS = 8` with `BATCH_SIZE = 100_000`** → still OOM. Per-worker pandas + pyarrow buffers blew past the per-worker budget, and **OS page cache for in-flight `.tmp` files** also counts in cgroup v2 `memory.current`.
  - **Fix:** dropped `BATCH_SIZE` to 25,000 and added explicit `del` + `gc.collect()` between batches (Cell 5).

### 3. Algorithm replacement — kept the contract, swapped the internals

Even at safe worker counts, the original SimHash was projected to take ~14 h. With the deadline at end-of-sprint, the function was rewritten while keeping Matty's contract (text-only, append-only, deterministic 64-bit fingerprint per doc):

| | original | this notebook |
| --- | --- | --- |
| shingling | char 3-grams (~12k features/doc) | word 5-grams (~3k features/doc — Charikar 2002 default) |
| per-shingle hash | `hashlib.blake2b` (cryptographic) | `xxhash.xxh64_intdigest` (~10× faster, uniform 64-bit) |
| bit accumulator | 64-iteration Python loop per shingle | one vectorised numpy broadcast |
| projected runtime | ~14 h | **finished in 1 h 55 min** |

The `hash` column (SHA-1) is unchanged from the original.

### 4. Worker-count tuning

- `NUM_WORKERS = 8` after the algorithm swap → container memory peaked at **31.07 GB / 32 GB** (page cache of six 1.4 GB `.tmp` files dominated).
- Backed off to **`NUM_WORKERS = 6`** → stable run, completed in 1 h 55 min.

### Lessons that will affect the next dedup-applying notebook

- Per-worker budget on this container is roughly **3 GB worker RSS + 2 GB page cache per active `.tmp`**; 6 workers is the practical ceiling for shard sizes of 1.5–3 GB.
- `cgroup` memory limit, not `psutil.virtual_memory()`, is the authoritative ceiling.
- For SimHash on long web text, word k-grams + a non-cryptographic 64-bit hash + numpy bit accumulation is ~100× faster than char-n-grams + cryptographic hash + Python bit loops, with no quality loss for near-duplicate detection.